# 🤟 SignTalk - AI Model Training & TFLite Quantization (Google Colab)
This notebook trains the **SignTalk Bidirectional LSTM Neural Network** on gesture keypoint sequence datasets and exports quantized TFLite models for deployment on Flutter mobile apps ($0 Budget).

## 1. Install & Import Dependencies

In [ ]:
!pip install mediapipe opencv-python tensorflow scikit-learn matplotlib seaborn
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

## 2. Load Gesture Sequence Keypoints Data

In [ ]:
# If data is uploaded via Zip or Drive, specify DATA_PATH here:
DATA_PATH = 'data'
ACTIONS = ['hello', 'thanks', 'help', 'doctor', 'water', 'food', 'police', 'yes', 'no', 'emergency']
SEQUENCE_LENGTH = 30
FEATURE_DIM = 126

def generate_synthetic_if_missing():
    if not os.path.exists(DATA_PATH):
        print("Generating sample dataset for training testing...")
        for action in ACTIONS:
            for seq in range(30):
                seq_dir = os.path.join(DATA_PATH, action, str(seq))
                os.makedirs(seq_dir, exist_ok=True)
                base = np.random.uniform(-0.5, 0.5, size=(FEATURE_DIM,))
                for frame in range(SEQUENCE_LENGTH):
                    noise = np.random.normal(0, 0.02, size=(FEATURE_DIM,))
                    np.save(os.path.join(seq_dir, f"{frame}.npy"), base + (frame/30)*0.15 + noise)

generate_synthetic_if_missing()

# Load sequences
sequences, labels = [], []
label_map = {action: i for i, action in enumerate(ACTIONS)}

for action in ACTIONS:
    action_dir = os.path.join(DATA_PATH, action)
    for seq in os.listdir(action_dir):
        seq_dir = os.path.join(action_dir, seq)
        if os.path.isdir(seq_dir):
            window = [np.load(os.path.join(seq_dir, f"{f}.npy")) for f in range(SEQUENCE_LENGTH)]
            sequences.append(window)
            labels.append(label_map[action])

X = np.array(sequences, dtype=np.float32)
y = tf.keras.utils.to_categorical(labels).astype(np.float32)
print(f"Loaded Data | Matrix X: {X.shape} | Labels y: {y.shape}")

## 3. Build & Train Bidirectional LSTM Architecture

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

model = tf.keras.models.Sequential([
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=True, activation='tanh'), input_shape=(30, 126)),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32, return_sequences=False, activation='tanh')),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(len(ACTIONS), activation='softmax')
])

model.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss='categorical_crossentropy', metrics=['categorical_accuracy'])
model.summary()

history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=60, batch_size=16)
model.save('signtalk_model.keras')

## 4. Convert & Quantize Model to TFLite

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
converter._experimental_lower_tensor_list_ops = False

tflite_model = converter.convert()
with open('signtalk_model.tflite', 'wb') as f:
    f.write(tflite_model)

with open('labels.txt', 'w', encoding='utf-8') as f:
    for a in ACTIONS:
        f.write(f"{a}\n")

print(f"TFLite Model saved! Size: {os.path.getsize('signtalk_model.tflite')/1024:.2f} KB")